# data_002  –  KIER 에너지 기초 통계 분석 및 시각화
`KIER_M02-04-01 Visualization` + `KIER_M02-04-02 Stats` 통합

## 01. Init

In [ ]:
#region Basic_Import
## Basic
import os
import sys

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import random

## Datetime
import datetime as dt
from datetime import datetime, date, timedelta

import glob

## 시각화
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.figsize'] = [10, 8]

from scipy import stats

# K-Means 알고리즘
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.model_selection import train_test_split

# CLustering 알고리즘의 성능 평가 측도
from sklearn.metrics import homogeneity_score, completeness_score, v_measure_score, adjusted_rand_score, silhouette_score, rand_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics.cluster import contingency_matrix

## 정규화
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn import metrics

## Init.
pd.options.display.float_format = '{:.10f}'.format
#endregion Basic_Import

In [ ]:
# DL 라이브러리 (분석 노트북 – 불필요)
# 모델링 노트북(M02-03_*)에서만 사용

In [ ]:
# ML/DL 모델 import (분석 노트북 – 불필요)
# 모델링 노트북(M02-03_*)에서만 사용

In [ ]:
## Import_Local
from core import data_datetime    as com_date
from core import data_analysis    as com_Analysis
from core import data_preprocessing as com_Prep
from core import data_visualization as com_Visual
from core import provider_kier_m02  as com_KIER_M02
from core import provider_kma       as com_ASOS

In [ ]:
## Init_config
SEED = 42

np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)

In [ ]:
## Define Todate str
str_now_ymd = dt.datetime.now().date()
str_now_y = dt.datetime.now().year
str_now_m = dt.datetime.now().month
str_now_d = dt.datetime.now().day
str_now_hr = dt.datetime.now().hour
str_now_min = dt.datetime.now().minute

print(dt.datetime.now())
print(str(str_now_y) + " / " + str(str_now_m)  + " / " + str(str_now_d))
print(str(str_now_hr) + " : " + str(str_now_min))

## 02. 데이터 로드

In [ ]:
## Dict_Domain  (provider_kier_m02 사용)
int_domain = 0   # 0=ELEC 1=HEAT 2=WATER 3=HOT_HEAT 4=HOT_FLOW 5=GAS
str_domain, str_col_accu, str_col_inst = com_KIER_M02.create_domain_str(int_domain)
_, str_dir_raw, _, str_dirName_bld, str_dirName_h = com_KIER_M02.create_dir_str(str_domain)

In [ ]:
## KMA_ASOS Data
str_dir_kmaAsos = "../data/data_KMA_ASOS/"

## Interpolate / Filled ASOS Data
str_file = 'ASOS_119_2010-2024_HR_INTP.csv'
Data_ASOS = pd.read_csv(str_dir_kmaAsos + str_file
                        , index_col = 0)
Data_ASOS['METER_DATE'] = pd.to_datetime(Data_ASOS['METER_DATE'])
Data_ASOS

In [ ]:
## Cluster 기준 Interval
# ▶ STATS 기준 (interval 명 표준화: 1D/1W/1M)
list_interval = ['10MIN', '1H', '1D', '1W', '1M']
str_interval = list_interval[2]   # 0=10min 1=1H 2=1D 3=1W 4=1M
K = 3  # 클러스터 수 (clustering_001에서 결정)
# ▶ VIZ 기준으로 실행 시: str_interval = list_interval[0]

str_file_clustering = 'KIER_' + str_domain + '_Labeled_' + str_interval + '.csv'
df_kier_h_cluster = pd.read_csv(str_dirName_h + str_file_clustering
                                , index_col=0).rename(columns={'index':'h_index'})[['h_index', 'target_' + str_domain]]
print(str_interval)
print(df_kier_h_cluster['target_' + str_domain].drop_duplicates())
df_kier_h_cluster

In [ ]:
list_kier_h_all = df_kier_h_cluster['h_index']
print(len(list_kier_h_all))
list_kier_h_c0 = df_kier_h_cluster[df_kier_h_cluster['target_' + str_domain] == 0]['h_index']
print(len(list_kier_h_c0))
list_kier_h_c1 = df_kier_h_cluster[df_kier_h_cluster['target_' + str_domain] == 1]['h_index']
print(len(list_kier_h_c1))
list_kier_h_c2 = df_kier_h_cluster[df_kier_h_cluster['target_' + str_domain] == 2]['h_index']
print(len(list_kier_h_c2))

In [ ]:
## 사용량 Data Load  (1시간 단위)
# ▶ STATS 기준 파일
str_file = 'KIER_' + str_domain + '_INST_03-02_1H.csv'
# ▶ VIZ 기준 파일
# str_file = 'KIER_' + str_domain + '_INST_1H_InstBaseUpdated.csv'

df_raw = pd.read_csv(str_dirName_h + str_file, index_col=0)
df_raw = df_raw.drop(columns=['METER_DATE'])
df_raw

In [ ]:
## 전체 사용량
df_kier_h_all = df_raw.copy()
## 0으로 표기되는 마지막행 제거
df_kier_h_all = df_kier_h_all[:-1]
print(df_kier_h_all.shape)

## Cluster별 사용량
## ■ C00
df_kier_h_c0 = df_raw.copy()[list_kier_h_c0]
df_kier_h_c0 = df_kier_h_c0[:-1]
print(df_kier_h_c0.shape)

## ■ C01
df_kier_h_c1 = df_raw.copy()[list_kier_h_c1]
df_kier_h_c1 = df_kier_h_c1[:-1]
print(df_kier_h_c1.shape)

## ■ C02
df_kier_h_c2 = df_raw.copy()[list_kier_h_c2]
df_kier_h_c2 = df_kier_h_c2[:-1]
print(df_kier_h_c2.shape)

In [ ]:
df_kier_h_c0

## 03. 시각화

분석 대상 세대를 선택 후 t-SNE 분포 시각화

In [ ]:
## ▶ 분석 대상 세대 선택
# 모든 세대
# df_raw = df_kier_h_all;  str_col_tar = str_domain + '_INST_SUM_ALL'
# C0 세대
# df_raw = df_kier_h_c0;   str_col_tar = str_domain + '_INST_SUM_C0'
## C1 세대 (VIZ 기준 기본)
df_raw = df_kier_h_c1
str_col_tar = str_domain + '_INST_SUM_C1'
# C2 세대
# df_raw = df_kier_h_c2;   str_col_tar = str_domain + '_INST_SUM_C2'

In [ ]:
## 군집 내 모든 값의 분산
df_concatnated = pd.DataFrame()
int_cnt = 0
for tar_col in df_raw.columns:
    # print(tar_col)
    df_temp = df_raw[[tar_col]].copy()
    df_temp = df_temp.rename(columns = {tar_col : "Usage"})

    if int_cnt == 0:
        df_concatnated = df_temp.copy()
    else:
        df_concatnated = pd.concat([df_concatnated, df_temp])

    int_cnt = int_cnt + 1

# print("Var : ", np.var(df_concatnated["Usage"]))
print("Max : ", np.max(df_concatnated["Usage"]))
print("75 : ", np.percentile(df_concatnated["Usage"], 75))
# print("Mean : ", np.mean(df_concatnated["Usage"])) ## 이미 아래 기술통계 항목에 평균이 포함되어있음
print("Med : ", np.median(df_concatnated["Usage"]))
print("25 : ", np.percentile(df_concatnated["Usage"], 25))
print("Min : ", np.min(df_concatnated["Usage"]))


print(stats.describe(df_concatnated))
print("Std : ", np.std(df_concatnated["Usage"]))
print("CV : ", np.std(df_concatnated["Usage"]) / np.mean(df_concatnated["Usage"]))

In [ ]:
df_raw.isna().sum()

In [ ]:
from sklearn.manifold import TSNE

## 표현할 데이터 차원
n_components = 2

## t-sne Model
model = TSNE(n_components = n_components)

## 학습 결과
res_tsne = model.fit_transform(df_raw)
print(res_tsne)

In [ ]:
print(res_tsne[0])

In [ ]:
if n_components == 2:
    ## 2차원
    plt.scatter(res_tsne[:, 0], res_tsne[:, 1])
elif n_components == 3:
    ## 3차원
    plt.scatter(res_tsne[:, 0], res_tsne[:, 1], res_tsne[:, 2])

plt.title(str(str_col_tar + " (n = " + str(n_components) + ")"))

## 비교를 위한 범위 지정
plt.xlim([-100, 100])
plt.ylim([-100, 100])
plt.show()

## 04. 기초 통계 분석

클러스터별 사용량 분포 · 기술통계 · 이상치 확인

In [ ]:
## ▶ 분석 대상 세대 선택
## C0 세대 (STATS 기준 기본)
df_raw = df_kier_h_c0
str_col_tar = str_domain + '_INST_SUM_C0'
# C1 세대
# df_raw = df_kier_h_c1;   str_col_tar = str_domain + '_INST_SUM_C1'
# C2 세대
# df_raw = df_kier_h_c2;   str_col_tar = str_domain + '_INST_SUM_C2'
# 모든 세대
# df_raw = df_kier_h_all;  str_col_tar = str_domain + '_INST_SUM_ALL'

In [ ]:
## 군집 내 모든 값의 분산
df_concatnated = pd.DataFrame()
int_cnt = 0
for tar_col in df_raw.columns:
    # print(tar_col)
    df_temp = df_raw[[tar_col]].copy()
    df_temp = df_temp.rename(columns = {tar_col : "Usage"})

    if int_cnt == 0:
        df_concatnated = df_temp.copy()
    else:
        df_concatnated = pd.concat([df_concatnated, df_temp])

    int_cnt = int_cnt + 1

# print("Var : ", np.var(df_concatnated["Usage"]))
print("Max : ", np.max(df_concatnated["Usage"]))
print("75 : ", np.percentile(df_concatnated["Usage"], 75))
# print("Mean : ", np.mean(df_concatnated["Usage"])) ## 이미 아래 기술통계 항목에 평균이 포함되어있음
print("Med : ", np.median(df_concatnated["Usage"]))
print("25 : ", np.percentile(df_concatnated["Usage"], 25))
print("Min : ", np.min(df_concatnated["Usage"]))


print(stats.describe(df_concatnated))
print("Std : ", np.std(df_concatnated["Usage"]))
print("CV : ", np.std(df_concatnated["Usage"]) / np.mean(df_concatnated["Usage"]))

In [ ]:
df_raw.isna().sum()

In [ ]:
from sklearn.manifold import TSNE

## 표현할 데이터 차원
n_components = 2

## t-sne Model
model = TSNE(n_components = n_components)

## 학습 결과
res_tsne = model.fit_transform(df_raw)
print(res_tsne)

In [ ]:
print(res_tsne[0])

In [ ]:
if n_components == 2:
    ## 2차원
    plt.scatter(res_tsne[:, 0], res_tsne[:, 1])
elif n_components == 3:
    ## 3차원
    plt.scatter(res_tsne[:, 0], res_tsne[:, 1], res_tsne[:, 2])

plt.title(str(str_col_tar + " (n = " + str(n_components) + ")"))

## 비교를 위한 범위 지정
plt.xlim([-100, 100])
plt.ylim([-100, 100])
plt.show()